# TF-STGNN — flood early warning for Sri Lanka

Trains and evaluates the model from `docs/PROJECT_PROPOSAL.md` §7.7, plus the
four baselines of §7.8 and the M0 → M6 ladder of §8.

## Before you hit *Save & Run All*

In the right-hand panel:

1. **Add Input** → attach both datasets:
   - `uom230429e/sri-lanka-flood-tabular-graph-2003-2025` — required by every stage
   - `uom230429e/flood-data-set` — required by the `sar` stage only
2. **Accelerator** → **GPU T4 ×2** (or P100)
3. **Internet** → **On** — needed to clone the code

## What it does

Stages run cheapest-and-most-decisive first, so a session that runs out of time
has still produced the results that matter. Anything that does not fit inside
`--time-budget-hours` is skipped with the exact command to finish it later, and
a stage that crashes does not take the others down with it.

| Stage | Answers | Rough cost |
|---|---|---|
| `baselines` | the bar to clear | ~10 min |
| `ladder` (M0→M5) | RQ1, RQ3, RQ4 | 2–4 h |
| `leakage` | **RQ2** | ~30 min |
| `spatial` | spatial generalisation | ~30 min |
| `sar` (M6) | **RQ5** | 3–5 h |

The full set exceeds one GPU session. The default budget of 8 h will complete
everything except `sar`; run that in a second session (see the last cell).

In [ ]:
# --- get the code -----------------------------------------------------
# rmtree rather than !rm -rf: this runs in the notebook process, so a re-run
# always starts from a clean checkout instead of failing on an existing dir.
import shutil, subprocess, sys

REPO = "https://github.com/heshannethmina/Srilanka-Flood-Data-Set-Creation"
DEST = "/kaggle/working/repo"

shutil.rmtree(DEST, ignore_errors=True)
subprocess.run(["git", "clone", "-q", REPO, DEST], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)
!ls {DEST}/model

In [ ]:
# --- environment check ------------------------------------------------
# Fails loudly *now* rather than three hours into the ladder.
import glob, os, torch

print(f"torch {torch.__version__} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Accelerator'}")

for name, need in [("flood_dataset.parquet", True), ("image_dataset.csv", False)]:
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    print(f"{name:24s} {'OK  ' + os.path.dirname(hits[0]) if hits else 'MISSING'}")
    assert hits or not need, f"attach the dataset containing {name} via 'Add Input'"

In [ ]:
# --- run everything that fits in the session --------------------------
!python /kaggle/working/repo/model/kaggle_run.py --stage all --time-budget-hours 8

In [ ]:
# --- package the results ----------------------------------------------
# /kaggle/working is the notebook's output, so this survives 'Save Version'.
import glob, json, os

files = sorted(glob.glob("/kaggle/working/runs/*"))
print(f"{len(files)} result files:")
for f in files:
    print(f"  {os.path.basename(f):40s} {os.path.getsize(f) / 1e3:8.1f} kB")

if files:
    !cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip

## Finishing the SAR stage

`sar` needs 3–5 h and will normally be skipped by the 8 h budget. Run it in a
second session — a fresh notebook with the same inputs, cells 1–2, then:

```python
!python /kaggle/working/repo/model/kaggle_run.py --stage sar
```

If it runs out of memory, drop the frame size and batch:

```python
!python /kaggle/working/repo/model/kaggle_run.py --stage sar --image-px 128 --batch-size 4
```

## Reading the output

Each stage prints a table covering every run so far. Judge the models on
**`ev.det`** — the share of flood episodes warned about *before* onset — at a
comparable FAR, not on PR-AUC. `target_flood_1d` is tomorrow's discharge
exceeding the 98th percentile, and discharge is strongly autocorrelated, so the
`discharge_pctl` baseline already ranks near the ceiling; pre-onset warning is
where the headroom is.

Plain accuracy is never reported: at a 1.9 % positive rate, "never floods"
scores 98.1 %.